In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

In [ ]:
# Load the entire JSON file
with open("../data/raw/app_1903340.json", 'r', encoding='utf-8') as f:
    app_data = pd.read_json(f, typ='series')

# Extract app metadata (everything except reviews)
app_info = app_data.drop('reviews')

# Extract reviews into a DataFrame
df = pd.json_normalize(app_data['reviews'])

# Convert timestamps to human-readable datetime format
df['timestamp_created'] = pd.to_datetime(df['timestamp_created'], unit='s')
df['timestamp_updated'] = pd.to_datetime(df['timestamp_updated'], unit='s')
df['author.last_played'] = pd.to_datetime(df['author.last_played'], unit='s')
df = df.sort_values('votes_funny', ascending=False)

# Display app info
print(f"App: {app_info['name']}")
print(f"App ID: {app_info['steam_appid']}")
print(f"Total reviews loaded: {len(df)}")


In [ ]:
df.head(20)

In [ ]:
df.tail(20)

In [ ]:
df["votes_funny"].describe()

In [ ]:


# Get all JSON files from data/raw directory
json_files = glob.glob("../data/raw/app_*.json")

# Create a list to store summary data
summary_data = []

# Process each game
for json_file in json_files:
    # Load the entire JSON file
    with open(json_file, 'r', encoding='utf-8') as f:
        app_data = pd.read_json(f, typ='series')
    
    # Extract reviews into a DataFrame
    reviews_df = pd.json_normalize(app_data['reviews'])
    
    # Calculate statistics
    summary = {
        'app_id': app_data['steam_appid'],
        'game_name': app_data['name'],
        'total_reviews': len(reviews_df),
        'voted_up': reviews_df['voted_up'].sum(),
        'voted_down': (~reviews_df['voted_up']).sum(),
        'max_votes_funny': reviews_df['votes_funny'].max()        
    }
    
    summary_data.append(summary)

# Create summary DataFrame
summary_df = pd.DataFrame(summary_data)
summary_df


## Additional Interesting Statistics

Other metrics worth exploring:
- **Review length vs funny votes**: Do longer reviews get more funny votes?
- **Playtime correlation**: Do people with more hours find different things funny?
- **Helpfulness vs humor**: Are funny reviews also helpful?
- **Temporal patterns**: When are the funniest reviews posted? (time of day, day of week, release proximity)
- **Positive vs negative humor**: Are negative reviews funnier than positive ones?
- **Language analysis**: What words/phrases appear in highly-voted funny reviews?


In [ ]:
# Timeline of funny reviews for the first game
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Scatter plot: When were funny reviews created?
axes[0, 0].scatter(df['timestamp_created'], df['votes_funny'], alpha=0.5, s=20)
axes[0, 0].set_xlabel('Review Date')
axes[0, 0].set_ylabel('Votes Funny')
axes[0, 0].set_title('Funny Votes Over Time')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Distribution of funny votes (histogram)
axes[0, 1].hist(df['votes_funny'], bins=50, edgecolor='black')
axes[0, 1].set_xlabel('Votes Funny')
axes[0, 1].set_ylabel('Number of Reviews')
axes[0, 1].set_title('Distribution of Funny Votes')
axes[0, 1].set_yscale('log')

# 3. Reviews by month
df['month'] = df['timestamp_created'].dt.to_period('M')
monthly_funny = df.groupby('month')['votes_funny'].agg(['count', 'sum', 'mean'])
monthly_funny.index = monthly_funny.index.to_timestamp()
axes[1, 0].plot(monthly_funny.index, monthly_funny['sum'], marker='o')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Total Funny Votes')
axes[1, 0].set_title('Total Funny Votes per Month')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Positive vs Negative reviews - which get more funny votes?
voted_comparison = df.groupby('voted_up')['votes_funny'].mean()
axes[1, 1].bar(['Negative', 'Positive'], [voted_comparison[False], voted_comparison[True]])
axes[1, 1].set_ylabel('Average Votes Funny')
axes[1, 1].set_title('Positive vs Negative Reviews: Average Funny Votes')

plt.tight_layout()
plt.show()

print(f"\nGame: {app_info['name']}")
print(f"Reviews with funny votes (>0): {(df['votes_funny'] > 0).sum()} / {len(df)}")
print(f"Average funny votes (all reviews): {df['votes_funny'].mean():.2f}")
print(f"Average funny votes (reviews with >0): {df[df['votes_funny'] > 0]['votes_funny'].mean():.2f}")


In [ ]:
# Calculate review age in days
df['review_age_days'] = (pd.Timestamp.now() - df['timestamp_created']).dt.days

# Create color mapping: green if funny votes > 0, red otherwise
df['is_funny'] = df['votes_funny'] > 0

# Create the scatter plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Age vs Helpful votes with funny votes color scale
scatter = axes[0, 0].scatter(df['review_age_days'], df['votes_up'], 
                             c=df['votes_funny'], cmap='viridis', alpha=0.6, s=30, 
                             norm=plt.Normalize(vmin=0, vmax=df['votes_funny'].quantile(0.95)))
axes[0, 0].set_xlabel('Review Age (days)')
axes[0, 0].set_ylabel('Helpful Votes')
axes[0, 0].set_title('Review Age vs Helpful Votes (Color = Funny Votes)')
axes[0, 0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[0, 0], label='Funny Votes')

# 2. Helpful votes vs Funny votes correlation
axes[0, 1].scatter(df['votes_up'], df['votes_funny'], alpha=0.5, s=20)
axes[0, 1].set_xlabel('Helpful Votes')
axes[0, 1].set_ylabel('Funny Votes')
axes[0, 1].set_title('Helpful vs Funny Votes Correlation')
axes[0, 1].grid(True, alpha=0.3)

# 3. Review age distribution by funny/not funny
df[df['is_funny']]['review_age_days'].hist(ax=axes[1, 0], bins=50, alpha=0.7, 
                                            color='green', label='Funny Reviews', edgecolor='black')
df[~df['is_funny']]['review_age_days'].hist(ax=axes[1, 0], bins=50, alpha=0.7, 
                                             color='red', label='Not Funny Reviews', edgecolor='black')
axes[1, 0].set_xlabel('Review Age (days)')
axes[1, 0].set_ylabel('Number of Reviews')
axes[1, 0].set_title('Age Distribution: Funny vs Not Funny Reviews')
axes[1, 0].legend()

# 4. Average helpful votes by age bins
df['age_bin'] = pd.cut(df['review_age_days'], bins=10)
age_analysis = df.groupby(['age_bin', 'is_funny'])['votes_up'].mean().unstack()
age_analysis.plot(kind='bar', ax=axes[1, 1], color=['red', 'green'], alpha=0.7)
axes[1, 1].set_xlabel('Review Age Range (days)')
axes[1, 1].set_ylabel('Average Helpful Votes')
axes[1, 1].set_title('Average Helpful Votes by Age and Funny Status')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].legend(['Not Funny', 'Funny'])

plt.tight_layout()
plt.show()


## Statistical Analysis: Age, Helpfulness, and Humor

In [ ]:
# Correlation analysis
print("=" * 60)
print("CORRELATION ANALYSIS")
print("=" * 60)
correlation_data = df[['review_age_days', 'votes_up', 'votes_funny']].corr()
print("\nCorrelation Matrix:")
print(correlation_data)

# Statistical comparison: Funny vs Not Funny reviews
print("\n" + "=" * 60)
print("FUNNY vs NOT FUNNY REVIEWS COMPARISON")
print("=" * 60)

comparison_stats = pd.DataFrame({
    'Funny Reviews': [
        df[df['is_funny']].shape[0],
        df[df['is_funny']]['votes_up'].mean(),
        df[df['is_funny']]['votes_up'].median(),
        df[df['is_funny']]['review_age_days'].mean(),
        df[df['is_funny']]['votes_funny'].mean()
    ],
    'Not Funny Reviews': [
        df[~df['is_funny']].shape[0],
        df[~df['is_funny']]['votes_up'].mean(),
        df[~df['is_funny']]['votes_up'].median(),
        df[~df['is_funny']]['review_age_days'].mean(),
        0
    ]
}, index=['Count', 'Avg Helpful Votes', 'Median Helpful Votes', 'Avg Age (days)', 'Avg Funny Votes'])

print(comparison_stats)

# Age-based analysis
print("\n" + "=" * 60)
print("AGE-BASED BREAKDOWN")
print("=" * 60)

age_categories = pd.cut(df['review_age_days'], 
                        bins=[0, 30, 90, 180, 365, float('inf')],
                        labels=['0-30 days', '31-90 days', '91-180 days', '181-365 days', '365+ days'])

age_breakdown = df.groupby(age_categories).agg({
    'recommendationid': 'count',
    'votes_up': 'mean',
    'votes_funny': 'mean',
    'is_funny': lambda x: (x.sum() / len(x) * 100)
}).round(2)

age_breakdown.columns = ['Review Count', 'Avg Helpful', 'Avg Funny', '% with Funny Votes']
print(age_breakdown)

# Top funny reviews analysis
print("\n" + "=" * 60)
print("TOP 10 FUNNY REVIEWS - CHARACTERISTICS")
print("=" * 60)

top_funny = df.nlargest(10, 'votes_funny')[['votes_funny', 'votes_up', 'review_age_days', 'voted_up']]
top_funny['voted_up'] = top_funny['voted_up'].map({True: 'Positive', False: 'Negative'})
print(top_funny)


In [ ]:
# Detailed cross-tabulation
print("=" * 60)
print("HELPFULNESS CATEGORIES vs FUNNY STATUS")
print("=" * 60)

# Create helpfulness categories
df['helpful_category'] = pd.cut(df['votes_up'], 
                                 bins=[-1, 0, 5, 20, 100, float('inf')],
                                 labels=['No Helpful', '1-5 Helpful', '6-20 Helpful', '21-100 Helpful', '100+ Helpful'])

crosstab = pd.crosstab(df['helpful_category'], df['is_funny'], 
                       values=df['votes_funny'], aggfunc='mean', margins=True)
crosstab.columns = ['Not Funny', 'Has Funny Votes', 'Total']
print("\nAverage Funny Votes by Helpfulness Category:")
print(crosstab.round(2))

# Distribution table
print("\n" + "=" * 60)
distribution_table = pd.crosstab(df['helpful_category'], df['is_funny'], margins=True)
distribution_table.columns = ['Not Funny', 'Has Funny Votes', 'Total']
print("\nCount Distribution:")
print(distribution_table)
